# 08 - Phase 1: data-leakage and duplicate analysis (CT dataset)

**Purpose**: determine whether duplicate images, near-duplicates or patient overlap make the current
CT evaluation overly optimistic.

**This notebook is entirely additive.** It does not modify or re-run notebooks 00-07, does not touch
`outputs/results_table.csv`, `experiments_log.csv` or `ablation_dropout.csv`, and does not overwrite
`faithful_split.csv` or `clean_split.csv`. Everything it writes goes to **`outputs/leakage/`** plus one
new split file, `outputs/splits/leakage_controlled_split.csv`.

**CPU only. This notebook trains nothing.** Steps 1-4 and 6-7 need only numpy/pandas/PIL - no
TensorFlow. Step 5 loads an existing checkpoint if one exists and evaluates it; if none exists it
**stops and reports** rather than retraining.

**Prerequisites**: the CT dataset present (`Data/`, or attached on Kaggle). Notebook 00 does *not*
have to have been run - this notebook indexes and hashes the images itself.

### Expected cost

| Step | Work | Cost |
|---|---|---|
| 1 | filename / metadata / DICOM inspection | seconds |
| 2 | MD5 over 1,000 files | ~5-15 s |
| 3 | pHash 1,000 images + 500k pair comparisons | ~30-60 s |
| 4 | union-find + greedy grouped split | < 1 s |
| 5 | evaluate one checkpoint (if any exists) | ~10-20 s, or skipped |
| 6-7 | write artefacts | < 1 s |

### What this analysis can and cannot establish

It can bound **duplicate-driven** leakage exactly. It **cannot** bound patient-driven leakage unless
genuine patient identifiers turn out to exist - step 1 decides that, and the rest of the notebook
adapts its claims accordingly.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import json
import numpy as np
import pandas as pd

from src.config import *
from src.data_utils import resolve_data_root, index_dataset, class_counts
from src.leakage_utils import (ensure_leakage_dir, LEAKAGE_DIR, PHASH_THRESHOLD,
                               inspect_patient_identifiers, ensure_md5,
                               exact_duplicate_groups, duplicate_pair_matrix,
                               summarise_pair_matrix, add_phashes,
                               near_duplicate_pairs, phash_threshold_sweep,
                               build_image_groups, group_summary,
                               build_leakage_controlled_split, split_class_table,
                               save_leakage_controlled_split, assert_no_group_leakage,
                               checkpoint_availability_report,
                               evaluate_checkpoint_on_split,
                               metrics_from_saved_predictions, comparison_table,
                               build_leakage_report_rows, write_leakage_outputs)

pd.set_option('display.width', 200)
ensure_dirs()
print('leakage artefacts ->', ensure_leakage_dir())
print('data root         ->', resolve_data_root())

In [ ]:
# Index the dataset. `orig_split` is the CURRENT train/val/test assignment - the
# split every existing CT result was produced under - so it is what the leakage
# questions are asked about.
df = index_dataset()
df = df.rename(columns={'orig_split': 'split'})   # local alias; file on disk untouched

print('images indexed:', len(df))
print()
print(df.groupby(['class', 'split']).size().unstack(fill_value=0)
        .reindex(CLASS_NAMES).to_string())
print()
print('split totals:', df['split'].value_counts().to_dict())

## STEP 1 - Patient information

Inspecting filenames, folder structure, sidecar CSV/JSON metadata and DICOM headers.

**The rule this step obeys**: a numeric filename stem is an *image index* until something external
proves otherwise. Treating it as a patient ID and then reporting "patient-level splitting" would
manufacture a guarantee that does not exist - the precise failure this phase is meant to detect.

**Looks right**: for this dataset, no metadata files, no DICOMs, and no identifier column - i.e.
`patient_ids_available = False`.

In [ ]:
patient_info = inspect_patient_identifiers(df, data_root=resolve_data_root())

print('images                    :', patient_info['n_images'])
print('distinct filename stems   :', patient_info['n_distinct_filename_stems'],
      f"(ratio {patient_info['stem_reuse_ratio']})")
print('filename examples         :', patient_info['filename_examples'][:5])
print('filename pattern hits     :', patient_info['filename_pattern_hits'])
print('columns available         :', patient_info['columns_available'])
print('identifier columns found  :', patient_info['identifier_columns_found'] or 'NONE')
print('metadata files found      :', patient_info['metadata_scan'].get('n_metadata_files'))
print('DICOM files found         :', patient_info['metadata_scan'].get('n_dicom_files'))
print('DICOM probe               :', patient_info['dicom_probe'])
print()
print('PATIENT IDS AVAILABLE     :', patient_info['patient_ids_available'])
print()
print(patient_info['verdict'])

## STEP 2 - Exact duplicates (MD5 over file bytes)

Byte-identical files. This is exact content identity, not similarity.

The six numbers that matter are the pair counts by split-pair. **Within-split duplicates are
redundancy** - they waste capacity but do not inflate the test score. **Cross-split duplicates are
leakage** - the model can memorise an image in training and be rewarded for it at test time. The
train-test cell is the one that directly inflates the reported number.

In [ ]:
df = ensure_md5(df)

n_images = len(df)
n_unique = int(df['hash'].nunique())
dup_groups_df = exact_duplicate_groups(df)
n_groups = int(dup_groups_df['group_id'].nunique()) if len(dup_groups_df) else 0
n_files_in_groups = int(len(dup_groups_df))
n_redundant = n_files_in_groups - n_groups

print(f'total images              : {n_images}')
print(f'unique images (MD5)       : {n_unique}')
print(f'duplicate groups          : {n_groups}')
print(f'files inside those groups : {n_files_in_groups}')
print(f'redundant files           : {n_redundant}  (removable with no information loss)')
print()
print('duplicate files per class:')
print(dup_groups_df['class'].value_counts().reindex(CLASS_NAMES, fill_value=0).to_string()
      if len(dup_groups_df) else '  none')

In [ ]:
# The six split-pair counts. Rows/cols are splits; cell = number of duplicate PAIRS.
pair_mat = duplicate_pair_matrix(df, split_col='split', group_col='hash')
print('duplicate PAIRS by split pair:')
print(pair_mat.to_string())

exact_pairs = summarise_pair_matrix(pair_mat)
print()
print('within-split (redundancy) :', exact_pairs['within_split_pairs'],
      '-> total', exact_pairs['total_within'])
print('cross-split  (LEAKAGE)    :', exact_pairs['cross_split_pairs'],
      '-> total', exact_pairs['total_cross'])
print()
n_files_cross = int(dup_groups_df[dup_groups_df['n_splits_spanned'] > 1]['filepath'].nunique()) \
                if len(dup_groups_df) else 0
print(f'FILES involved in cross-split duplicate groups: {n_files_cross}')
if exact_pairs['total_cross'] == 0:
    print('No exact cross-split leakage.')
else:
    print('Exact cross-split leakage IS present - quantified above.')

In [ ]:
# Which classes carry the cross-split leakage? Concentration matters: leakage in
# `normal` inflates tumour DETECTION, which is already the easy sub-task.
if len(dup_groups_df):
    cross = dup_groups_df[dup_groups_df['n_splits_spanned'] > 1]
    print('cross-split duplicate files per class:')
    print(cross['class'].value_counts().reindex(CLASS_NAMES, fill_value=0).to_string())
    print()
    print('split combinations spanned:')
    print(cross.groupby('splits_spanned')['group_id'].nunique().to_string())
    print()
    print('largest duplicate groups:')
    top = dup_groups_df.groupby('group_id').agg(
        size=('group_size', 'first'), cls=('class', 'first'),
        spans=('splits_spanned', 'first')).sort_values('size', ascending=False).head(8)
    print(top.to_string())
else:
    print('no duplicate groups')

## STEP 3 - Near duplicates (perceptual hash)

A 64-bit DCT pHash per image, then every pair within a Hamming distance of `PHASH_THRESHOLD` (5/64).
Exact duplicates are excluded from this table so it reports *additional* evidence beyond step 2.

**Why pHash and not neural embeddings**: pHash is one image decode plus a 32x32 DCT per file, and the
pair search is integer XOR + popcount. That is seconds on CPU for 1,000 images. A CNN-embedding
analysis would cost a forward pass per image and add a model dependency for no gain at this scale -
explicitly out of scope per the brief.

The threshold sweep below is printed so the choice of 5 is auditable rather than a magic number: if
the pair count is flat across 4-8, the finding is robust to the threshold; if it explodes, it is not.

**Two things worth inspecting by eye if they appear**: near-duplicate pairs that are *cross-split*
(leakage that MD5 misses) and pairs that are *cross-class* (near-identical images with different
labels, which would indicate a labelling problem rather than leakage).

In [ ]:
df = add_phashes(df)
print('pHash computed for', len(df), 'images')
print()
print('threshold sensitivity sweep:')
print(phash_threshold_sweep(df).to_string(index=False))

In [ ]:
near_df = near_duplicate_pairs(df, threshold=PHASH_THRESHOLD,
                               exclude_exact=True, split_col='split')

n_near = len(near_df)
n_near_cross_split = int(near_df['cross_split'].sum()) if n_near else 0
n_near_cross_class = int((~near_df['same_class']).sum()) if n_near else 0

print(f'near-duplicate pairs (Hamming <= {PHASH_THRESHOLD}, excluding exact duplicates): {n_near}')
print(f'  of which cross-split : {n_near_cross_split}   <- leakage MD5 does not catch')
print(f'  of which cross-class : {n_near_cross_class}   <- possible labelling issue, inspect')
print()
if n_near:
    print('closest pairs:')
    print(near_df.head(12)[['hamming_distance', 'cross_split', 'same_class',
                            'split_a', 'split_b', 'class_a', 'class_b',
                            'filename_a', 'filename_b']].to_string(index=False))
else:
    print('No near-duplicates beyond the exact duplicates already found.')

In [ ]:
# Image groups: connected components over (exact duplicate OR near duplicate).
# This is the unit that must not straddle a split boundary.
df = build_image_groups(df, near_df)
groups = group_summary(df)

for k, v in groups.items():
    print(f'{k}: {v}')
print()
print('NOTE: these are IMAGE groups, not patient groups. Two different slices from')
print('one patient that look different fall into different groups, so this bounds')
print('duplicate-driven leakage only.')

## STEP 4 - Leakage-controlled split

**Which kind of split this is depends on step 1**, and the notebook decides rather than assuming:

* patient IDs available -> a genuine **patient-level** grouped split;
* patient IDs unavailable -> an **image-group** split over duplicate/near-duplicate components,
  which is scientifically defensible and which the code labels
  `leakage_controlled_image_group` so no reader can mistake it for a patient-level guarantee.

Written to a **new** file, `outputs/splits/leakage_controlled_split.csv`. `save_leakage_controlled_split()`
refuses the names `faithful` and `clean`, so the originals cannot be overwritten.

In [ ]:
if patient_info['patient_ids_available']:
    GROUP_COL = 'patient_id'
    SPLIT_KIND = 'patient_level'
    print('Patient IDs exist -> building a PATIENT-LEVEL grouped split.')
else:
    GROUP_COL = 'image_group'
    SPLIT_KIND = 'image_group_level'
    print('No patient IDs -> building an IMAGE-GROUP leakage-controlled split.')
    print('Patient-level leakage remains UNVERIFIABLE and is not claimed anywhere.')

controlled = build_leakage_controlled_split(df, group_col=GROUP_COL)
assert_no_group_leakage(controlled, group_col=GROUP_COL)
print('\nno-group-leakage assertion: PASSED')

split_tbl = split_class_table(controlled)
print()
print(split_tbl.to_string())
print()
print('split totals:', controlled['split'].value_counts().to_dict())

In [ ]:
# Residual leakage check on the NEW split, using the same machinery as step 2.
new_pairs = summarise_pair_matrix(
    duplicate_pair_matrix(controlled, split_col='split', group_col='hash'))
print('exact duplicate pairs under the leakage-controlled split:')
print('  within-split:', new_pairs['within_split_pairs'], '-> total', new_pairs['total_within'])
print('  cross-split :', new_pairs['cross_split_pairs'], '-> total', new_pairs['total_cross'],
      '(must be 0)')

near_after = near_duplicate_pairs(controlled, threshold=PHASH_THRESHOLD,
                                 exclude_exact=True, split_col='split')
n_near_cross_after = int(near_after['cross_split'].sum()) if len(near_after) else 0
print('  near-duplicate cross-split pairs:', n_near_cross_after, '(must be 0)')

controlled_path = save_leakage_controlled_split(controlled, name='leakage_controlled')
print('\nsaved ->', controlled_path)
print('(faithful_split.csv and clean_split.csv untouched)')

## STEP 5 - Evaluation

The brief's condition: **if an existing checkpoint can be evaluated without retraining, do so; if
retraining is required, STOP and report.**

The gate below decides. Note the interpretation caveat that applies even when a checkpoint *is*
found: a model trained on the ORIGINAL split has very likely already seen images that now sit in the
leakage-controlled test split, so re-scoring it gives an **upper bound** on leakage-controlled
performance, not a clean estimate. Only retraining under the new split gives a clean number.

In [ ]:
ck = checkpoint_availability_report()
print('models dir              :', ck['models_dir'], '(exists:', ck['models_dir_exists'], ')')
print('checkpoints found       :', ck['n_checkpoints'])
for c in ck['checkpoints']:
    print('   ', c)
print('can evaluate w/o retrain:', ck['can_evaluate_without_retraining'])
print()
print(ck['note'])

In [ ]:
# ORIGINAL-split performance, recomputed from the already-saved prediction file.
# Zero compute, and it uses the identical code path as the controlled evaluation
# so the two columns of the comparison are strictly comparable.
ORIGINAL_RUN = 'miniconvnet_faithful'    # single run on the dataset's own test split
try:
    original_perf = metrics_from_saved_predictions(ORIGINAL_RUN)
    print(f"ORIGINAL ({ORIGINAL_RUN}, n={original_perf['n_eval']}):")
    for k, v in original_perf['metrics'].items():
        print(f'  {k}: {v:.4f}')
    print(f"  tumour detection: {original_perf['tumor_detection_accuracy']:.4f}")
    print(f"  subtype         : {original_perf['subtype_accuracy']:.4f}")
    print('  per-class F1    :', {k: round(v, 4) for k, v in original_perf['per_class_f1'].items()})
    print('  confusion matrix:')
    print(np.array(original_perf['confusion_matrix']))
except FileNotFoundError as exc:
    original_perf = None
    print('original predictions not found:', exc)

In [ ]:
controlled_perf = None
RETRAIN_REQUIRED = not ck['can_evaluate_without_retraining']

if RETRAIN_REQUIRED:
    print('=' * 78)
    print('STOP - LEAKAGE-CONTROLLED EVALUATION REQUIRES RETRAINING')
    print('=' * 78)
    print('No MiniConvNet checkpoint exists on disk, so the trained model cannot be')
    print('re-scored on the leakage-controlled split. models/ is git-ignored and no')
    print('weights were retained from any previous run.')
    print()
    print('This notebook will NOT train anything. To obtain a leakage-controlled')
    print('number, a retrain is required - approve it explicitly first.')
    print()
    print('Estimated CPU cost of the retrain (from measured s/epoch in the notes')
    print('column of outputs/experiments_log.csv, ~4-6 s/epoch for MiniConvNet):')
    print('  single run  , 60-epoch budget : ~4-6 min')
    print('  3-fold CV   , 40-epoch budget : ~10-12 min')
    print()
    print('Everything else in this notebook (steps 1-4, 6-7) has completed and its')
    print('artefacts are written regardless.')
else:
    CHECKPOINT = ck['checkpoints'][0]
    print('Evaluating existing checkpoint (loading weights only, no training):')
    print(' ', CHECKPOINT)
    controlled_perf = evaluate_checkpoint_on_split(
        CHECKPOINT, controlled, split='test',
        run_name='leakage_controlled_miniconvnet_eval')
    print()
    for k, v in controlled_perf['metrics'].items():
        print(f'  {k}: {v:.4f}')
    print(f"  tumour detection: {controlled_perf['tumor_detection_accuracy']:.4f}")
    print(f"  subtype         : {controlled_perf['subtype_accuracy']:.4f}")
    print('  status          :', controlled_perf['status'])
    print('  confusion matrix:')
    print(np.array(controlled_perf['confusion_matrix']))
    print()
    print('CAVEAT: this checkpoint was trained on the ORIGINAL split and has likely')
    print('seen part of this test set. Treat it as an UPPER BOUND, not a clean estimate.')

In [ ]:
cmp_tbl = comparison_table(original_perf, controlled_perf) if original_perf else pd.DataFrame()
if len(cmp_tbl):
    print('ORIGINAL vs LEAKAGE-CONTROLLED')
    print(cmp_tbl.to_string(index=False))
    if controlled_perf is None:
        print()
        print('leakage_controlled column is empty: no checkpoint to evaluate (see step 5).')
else:
    print('no comparison available')

## STEP 6 - Save outputs

Five artefacts, all new, all under `outputs/leakage/`. No existing result file is read for writing or
overwritten.

In [ ]:
exact_summary = {
    'n_images': n_images,
    'n_unique_hashes': n_unique,
    'n_duplicate_groups': n_groups,
    'n_files_in_duplicate_groups': n_files_in_groups,
    'n_redundant_files': n_redundant,
    'pairs': exact_pairs,
}
near_summary = {
    'threshold': PHASH_THRESHOLD,
    'n_pairs': n_near,
    'n_cross_split': n_near_cross_split,
    'n_cross_class': n_near_cross_class,
}

report_df = build_leakage_report_rows(patient_info, exact_summary, near_summary,
                                      groups, split_tbl)
print(report_df.to_string(index=False))

In [ ]:
summary = {
    'phase': 'PHASE 1 - data leakage / duplicate analysis',
    'dataset': 'Chest CT-Scan images (Kaggle, Mohamed Hany) - 4 classes',
    'analysis_split_analysed': 'faithful (the dataset original train/valid/test folders)',
    'total_images': n_images,
    'unique_images_md5': n_unique,
    'duplicate_groups': n_groups,
    'redundant_files': n_redundant,
    'cross_split_duplicate_pairs': exact_pairs['total_cross'],
    'cross_split_duplicate_pairs_detail': exact_pairs['cross_split_pairs'],
    'within_split_duplicate_pairs': exact_pairs['within_split_pairs'],
    'near_duplicate_threshold_hamming': PHASH_THRESHOLD,
    'near_duplicate_pairs_excluding_exact': n_near,
    'near_duplicate_cross_split_pairs': n_near_cross_split,
    'near_duplicate_cross_class_pairs': n_near_cross_class,
    'image_groups': groups,
    'patient_ids_available': patient_info['patient_ids_available'],
    'patient_level_split_claimable': patient_info['patient_ids_available'],
    'patient_id_verdict': patient_info['verdict'],
    'leakage_controlled_split_kind': SPLIT_KIND,
    'leakage_controlled_split_group_column': GROUP_COL,
    'leakage_controlled_split_path': str(controlled_path),
    'residual_cross_split_duplicate_pairs': new_pairs['total_cross'],
    'residual_cross_split_near_duplicate_pairs': n_near_cross_after,
    'original_performance': original_perf,
    'leakage_controlled_performance': controlled_perf,
    'retrain_required_for_controlled_evaluation': RETRAIN_REQUIRED,
    'checkpoint_report': ck,
    'cpu_only': True,
    'models_trained_in_this_notebook': 0,
}

paths = write_leakage_outputs(dup_groups_df, near_df, controlled, report_df, summary)
print('files written:')
for k, v in paths.items():
    print(f'  {k:24s} {v}')

## STEP 7 - Interpretation

The six questions from the brief, answered from the numbers above. Print, read, and copy into the
write-up - do not restate them more strongly than the printed values support.

In [ ]:
print('1. IS THERE EVIDENCE OF LEAKAGE?')
tot_leak = exact_pairs['total_cross'] + n_near_cross_split
print(f'   {"YES" if tot_leak else "NO"} - {exact_pairs["total_cross"]} exact cross-split pairs '
      f'+ {n_near_cross_split} near-duplicate cross-split pairs.')
print()
print('2. HOW MANY CROSS-SPLIT DUPLICATES?')
print(f'   {exact_pairs["total_cross"]} exact pairs: {exact_pairs["cross_split_pairs"]}')
print(f'   {n_files_cross} distinct files sit in duplicate groups spanning >1 split.')
print()
print('3. ARE NEAR-DUPLICATES PRESENT?')
print(f'   {n_near} pairs at Hamming <= {PHASH_THRESHOLD} beyond the exact duplicates '
      f'({n_near_cross_split} cross-split, {n_near_cross_class} cross-class).')
print()
print('4. ARE PATIENT IDS AVAILABLE?')
print(f'   {patient_info["patient_ids_available"]}')
print()
print('5. CAN PATIENT-AWARE EVALUATION BE CLAIMED?')
if patient_info['patient_ids_available']:
    print('   YES - genuine identifiers exist and the split is grouped on them.')
else:
    print('   NO. No patient identifiers exist, so patient-level leakage can be neither')
    print('   verified nor excluded. The split built here is IMAGE-GROUP level. Claiming')
    print('   "patient-aware evaluation" would be unsupported.')
print()
print('6. DOES PERFORMANCE CHANGE UNDER LEAKAGE-CONTROLLED EVALUATION?')
if controlled_perf is None:
    print('   NOT MEASURED - no checkpoint exists, and this notebook does not retrain.')
    print('   Requires an approved retrain (~4-6 min single run / ~10-12 min 3-fold CV on CPU).')
else:
    d = controlled_perf['metrics']['accuracy'] - original_perf['metrics']['accuracy']
    print(f'   accuracy {original_perf["metrics"]["accuracy"]:.4f} -> '
          f'{controlled_perf["metrics"]["accuracy"]:.4f} ({d:+.4f})')
    print('   Upper bound only - the checkpoint saw part of this test set in training.')

In [ ]:
# Final summary block, in the exact format requested.
def _fmt(p):
    return 'not measured' if p is None else f"acc={p['metrics']['accuracy']:.4f} " \
        f"f1={p['metrics']['f1_macro']:.4f} det={p['tumor_detection_accuracy']:.4f} " \
        f"sub={p['subtype_accuracy']:.4f}"

print('PHASE:                           Phase 1 - leakage / duplicate analysis')
print('DATASET:                         Chest CT-Scan images (Kaggle) - 4 classes')
print(f'TOTAL IMAGES:                    {n_images}')
print(f'UNIQUE IMAGES:                   {n_unique}')
print(f'DUPLICATE GROUPS:                {n_groups}')
print(f'CROSS-SPLIT DUPLICATES:          {exact_pairs["total_cross"]} pairs / {n_files_cross} files')
print(f'NEAR-DUPLICATES:                 {n_near} pairs (<= {PHASH_THRESHOLD}), '
      f'{n_near_cross_split} cross-split')
print(f'PATIENT IDS:                     {"available" if patient_info["patient_ids_available"] else "NOT AVAILABLE"}')
print(f'LEAKAGE STATUS:                  {"PRESENT" if tot_leak else "none detected"}')
print(f'ORIGINAL PERFORMANCE:            {_fmt(original_perf)}')
print(f'LEAKAGE-CONTROLLED PERFORMANCE:  {_fmt(controlled_perf)}')
print(f'FILES CREATED:                   {", ".join(sorted(os.path.basename(v) for v in paths.values()))}, '
      f'{os.path.basename(str(controlled_path))}')
print('CPU TIME:                        fill in from the cell timings above (no training performed)')
print('LIMITATIONS:                     no patient IDs -> patient-level leakage unverifiable; '
      'pHash threshold is a judgement call (sweep printed); '
      'controlled performance requires an approved retrain')
print('RESEARCH INTERPRETATION:         write from the step 7 answers above')